In [0]:
%run ../utils/adls_auth

In [0]:
fact_streaming = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/fact_trips_streaming")
print(f"fact_trips_streaming rows: {fact_streaming.count()}")

# Check partitioning
spark.sql("SHOW PARTITIONS delta.`abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/fact_trips_streaming`").show()

# Check audit columns
fact_streaming.select("event_key", "_batch_id", "_created_at", "_updated_at").show(5)

In [0]:
from pyspark.sql.functions import col, sha2, concat_ws, to_date
fact_trips = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/fact_trips")
print(f"fact_trips rows: {fact_trips.count():,}")

# Check partitioning
spark.sql("SHOW PARTITIONS delta.`abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/fact_trips`").show(10)

# Check audit columns
fact_trips.select("trip_key", "_batch_id", "_created_at", "_updated_at").show(5)

# Check SCD2 join worked (should have very few nulls)
null_pickups = fact_trips.filter(col("pickup_location_key").isNull()).count()
print(f"Null pickup_location_key: {null_pickups:,} (should match Phase 6 orphaned zones)")

In [0]:
dim_location = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_location")
print(f"dim_location rows: {dim_location.count()}")
print("Sample location_ids:")
dim_location.select("location_id", "borough", "zone", "is_current", "effective_start_date", "effective_end_date").show(10)

In [0]:
trips_silver = spark.read.format("delta").load("abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/trips_silver")
print(f"trips_silver rows: {trips_silver.count():,}")

# Check unique PULocationID values
trips_silver.select("PULocationID").distinct().orderBy("PULocationID").show(20)

# Check if any match dim_location
location_ids = dim_location.select("location_id").collect()
location_id_set = set([row.location_id for row in location_ids])
print(f"\nNumber of location_ids in dim_location: {len(location_id_set)}")

# Check what percentage of PULocationID values are in dim_location
trips_ids = trips_silver.select("PULocationID").distinct().collect()
trips_id_set = set([row.PULocationID for row in trips_ids])
matched = trips_id_set.intersection(location_id_set)
unmatched = trips_id_set - location_id_set
print(f"Trips PULocationID values: {len(trips_id_set)}")
print(f"Matched in dim_location: {len(matched)}")
print(f"Unmatched: {len(unmatched)}")
if unmatched:
    print(f"Example unmatched: {list(unmatched)[:10]}")

In [0]:
# Check if the date range join is working
from pyspark.sql.functions import col, count

test_join = (
    trips_silver.limit(1000)
    .join(
        dim_location.alias("pu"),
        (col("PULocationID") == col("pu.location_id")) &
        (col("pickup_date") >= col("pu.effective_start_date")) &
        (col("pickup_date") <= col("pu.effective_end_date")),
        "left",
    )
    .select(
        col("PULocationID"),
        col("pu.location_id"),
        col("pu.zone"),
        col("pickup_date"),
        col("pu.effective_start_date"),
        col("pu.effective_end_date"),
    )
)

test_join.show(20)

In [0]:
dim_weather = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_weather")
print(f"dim_weather rows: {dim_weather.count()}")  # Should be ~8,760
dim_weather.groupBy("weather_category").count().show()

In [0]:
dim_date = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_date")
print(f"dim_date rows: {dim_date.count()}")  # Should be 365
dim_date.show(5)

In [0]:
fact_trips = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/fact_trips")
null_pickups = fact_trips.filter(col("pickup_location_key").isNull()).count()
print(f"Null pickup_location_key: {null_pickups:,}")

In [0]:
# 1. Check all fact tables
fact_trips = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/fact_trips")
fact_streaming = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/fact_trips_streaming")

print("=" * 60)
print("PHASE 7 — GOLD LAYER VERIFICATION")
print("=" * 60)

# 2. Dimension counts
print("\n--- DIMENSIONS ---")
dim_date = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_date")
dim_location = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_location")
dim_vendor = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_vendor")
dim_weather = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_weather")

print(f"dim_date:          {dim_date.count():,} rows")
print(f"dim_location:      {dim_location.count():,} rows")
print(f"dim_vendor:        {dim_vendor.count():,} rows")
print(f"dim_weather:       {dim_weather.count():,} rows")

# 3. Fact counts
print("\n--- FACTS ---")
print(f"fact_trips:                    {fact_trips.count():,} rows")
print(f"fact_trips_streaming:          {fact_streaming.count():,} rows")

# 4. NULL key checks
print("\n--- DATA QUALITY ---")
null_pickups = fact_trips.filter(col("pickup_location_key").isNull()).count()
null_dropoffs = fact_trips.filter(col("dropoff_location_key").isNull()).count()
null_vendors = fact_trips.filter(col("vendor_key").isNull()).count()
null_weather = fact_trips.filter(col("weather_key").isNull()).count()

print(f"NULL pickup_location_key:  {null_pickups:,}")
print(f"NULL dropoff_location_key: {null_dropoffs:,}")
print(f"NULL vendor_key:           {null_vendors:,}")
print(f"NULL weather_key:          {null_weather:,}")

# 5. Partitioning
print("\n--- PARTITIONING ---")
print("fact_trips partitions (sample):")
spark.sql("SHOW PARTITIONS delta.`abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/fact_trips`").show(5)

print("\nfact_trips_streaming partitions:")
spark.sql("SHOW PARTITIONS delta.`abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/fact_trips_streaming`").show()

# 6. Audit columns
print("\n--- AUDIT COLUMNS (Sample) ---")
fact_trips.select("trip_key", "_batch_id", "_created_at", "_updated_at").show(5, truncate=False)

In [0]:
# Investigate NULL vendor_key
fact_trips.filter(col("vendor_key").isNull()).select("vendor_key").distinct().show()

# Check dim_vendor
dim_vendor = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_vendor")
dim_vendor.show()

In [0]:
# Check what VendorID values exist in trips_silver (not fact_trips)
trips_silver = spark.read.format("delta").load("abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/trips_silver")

# Check dim_vendor
dim_vendor = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_vendor")
vendor_ids = [row.vendor_id for row in dim_vendor.select("vendor_id").collect()]
print(f"Vendor IDs in dim_vendor: {vendor_ids}")

# Check trips with VendorID not in dim_vendor
bad_vendors = trips_silver.filter(~col("VendorID").isin(vendor_ids))
bad_vendors.groupBy("VendorID").count().orderBy(col("count").desc()).show()

# Check if VendorID has NULL values
trips_silver.filter(col("VendorID").isNull()).count()

In [0]:
from pyspark.sql.functions import col

# Count Vendor ID 6 rows in trips_silver
trips_silver = spark.read.format("delta").load("abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/trips_silver")
vendor_6_count = trips_silver.filter(col("VendorID") == 6).count()
print(f"Vendor ID 6 rows: {vendor_6_count:,}")

# Check distribution by month
trips_silver.filter(col("VendorID") == 6).groupBy("month").count().orderBy("month").show()

# Check if Vendor ID 6 has any other suspicious patterns
trips_silver.filter(col("VendorID") == 6).select(
    "payment_type", "fare_amount", "trip_distance"
).describe().show()

In [0]:
# Check NULL weather_key distribution
fact_trips = spark.read.format("delta").load("abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/fact_trips")
null_weather = fact_trips.filter(col("weather_key").isNull())
null_weather.groupBy("pickup_date_key").count().orderBy("pickup_date_key").show(20)

In [0]:

pipeline_log = spark.read.format("delta").load("abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_control/pipeline_run_log")
pipeline_log.orderBy("start_time", ascending=False).show(10)